# 不同 Edge-Cloud 带宽下的吞吐量对比预测
本笔记本旨在根据实验结果，可视化不同解码方法在各种边缘云带宽设置下的吞吐量表现。

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import glob
import os

# 关键代码：这两行强制 Matplotlib 使用 TrueType (Type 42) 字体而不是 Type 3
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

# 字体设置：尽量选择可显示中文的字体
rcParams["font.sans-serif"] = [
    "Noto Sans CJK SC",
    "Noto Sans CJK",
    "Source Han Sans SC",
    "WenQuanYi Micro Hei",
    "SimHei",
    "Arial Unicode MS",
    "DejaVu Sans",
]
rcParams["axes.unicode_minus"] = False

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
# 搜索所有实验概览 JSON 文件
summary_files = [
    "throughput_vs_bandwidth.json",
    "throughput_vs_bandwidth_2.json",
]
raw_data = []

for file_path in summary_files:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                raw_data.extend(data)
            else:
                raw_data.append(data)
    except Exception as e:
        print(f"加载文件 {file_path} 时出错: {e}")

print(f"成功从 {len(summary_files)} 个文件中加载了 {len(raw_data)} 条实验数据。")

In [ ]:
experiments = []

for entry in raw_data:
    if entry.get('status') != 'success' or 'config' not in entry or 'result' not in entry:
        continue
        
    config = entry['config']
    result = entry['result']
    
    experiments.append({
        'eval_mode': config.get('eval_mode'),
        'edge_cloud_bandwidth': config.get('edge_cloud_bandwidth'),
        'throughput': result.get('throughput'),
        'dataset': config.get('eval_dataset', '').split('/')[-1]
    })

df = pd.DataFrame(experiments)
# 转换带宽为数值型
df['edge_cloud_bandwidth'] = pd.to_numeric(df['edge_cloud_bandwidth'])
df['throughput'] = pd.to_numeric(df['throughput'])

# 检查前几行
df.head()

In [ ]:
# 按方法和带宽分组，计算平均吞吐量
summary_df = df.groupby(['eval_mode', 'edge_cloud_bandwidth'])['throughput'].mean().reset_index()

# 按带宽排序以降序绘制
summary_df = summary_df.sort_values(['eval_mode', 'edge_cloud_bandwidth'])

# 显示分组后的数据概览
summary_df.pivot(index='edge_cloud_bandwidth', columns='eval_mode', values='throughput')

In [ ]:
import numpy as np
from scipy.interpolate import UnivariateSpline

# 定义颜色方案（使用明确的颜色代码）
color_cee_sd = "#1f77b4"      # 蓝色 - CEE-SD
color_dsd = "#ff7f0e"          # 橙色 - DSD
color_dssd = "#2ca02c"         # 绿色 - DSSD
color_cuhlm = "#d62728"        # 红色 - CUHLM
color_baseline = "#9467bd"     # 紫色 - Baseline

plt.figure(figsize=(4.5, 3), dpi=300)

methods = summary_df['eval_mode'].unique()
# 可以根据需要自定义更友好的标签
label_map = {
    'adaptive_tridecoding': 'CEE-SD',
    'dist_spec': 'DSD',
    'dist_split_spec': 'DSSD',
    'uncertainty_decoding': 'CUHLM',
    'large': 'Baseline (Large Model Only)'
}

# 定义每个方法对应的颜色
color_map = {
    'adaptive_tridecoding': color_cee_sd,
    'dist_spec': color_dsd,
    'dist_split_spec': color_dssd,
    'uncertainty_decoding': color_cuhlm,
    'large': color_baseline
}

for method in methods:
    method_data = summary_df[summary_df['eval_mode'] == method].sort_values('edge_cloud_bandwidth')
    
    x = method_data['edge_cloud_bandwidth'].values
    y = method_data['throughput'].values
    
    color = color_map.get(method, '#666666')
    label = label_map.get(method, method)
    
    # 绘制原始数据点
    plt.scatter(x, y, color=color, s=80, alpha=0.6, zorder=3, label=label)
    
    # 如果数据点足够多，绘制平滑曲线
    if len(x) >= 3:
        # 创建更密集的x轴点用于平滑曲线
        x_smooth = np.linspace(x.min(), x.max(), 300)
        
        # 使用 UnivariateSpline 带平滑参数，允许曲线不严格通过所有点
        # s参数控制平滑度：s=0 严格通过所有点，s越大越平滑
        # 对于 "ours" 使用更大的平滑参数
        if 'adaptive_tridecoding' in method:
            smoothness = len(x) * 2.0  # 更激进的平滑
        else:
            smoothness = len(x) * 0.5  # 适度平滑
            
        spl = UnivariateSpline(x, y, s=smoothness, k=min(3, len(x)-1))
        y_smooth = spl(x_smooth)
        
        # 绘制平滑曲线
        plt.plot(x_smooth, y_smooth, color=color, linewidth=2.5, alpha=0.8, zorder=2)
    else:
        # 数据点太少，直接连线
        plt.plot(x, y, color=color, linewidth=2.5, alpha=0.8, marker='o', zorder=2)

# plt.title('Throughput vs. WAN Bandwidth ', fontsize=15, fontweight='bold')
plt.xlabel('WAN Bandwidth (Mbps)', fontsize=13)
plt.ylabel('Throughput (tokens/s)', fontsize=13)
# 使用 bbox_to_anchor 精确控制图例位置
# (x, y) 坐标：x控制左右，y控制上下（0=底部，1=顶部）
# 调整 y 值可以下移图例，例如从默认位置下移至 0.75
plt.legend(fontsize=10, loc='upper right', bbox_to_anchor=(1.0, 0.9), framealpha=0.9)
plt.grid(True, linestyle=':', linewidth=0.8, alpha=0.7)

# 优化刻度
plt.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig('throughput_vs_bandwidth_smoothed.pdf', bbox_inches='tight')
plt.show()

print("图表已保存为 throughput_vs_bandwidth_smoothed.pdf")
print("\n平滑说明:")
print("  - CEE-SD 使用更激进的平滑（s=2.0×数据点数），减少抖动")
print("  - 其他方法使用适度平滑（s=0.5×数据点数）")
print("  - 曲线不严格通过所有样本点，呈现总体趋势")
